<a href="https://colab.research.google.com/github/gowripreetham/SJSU_Deep_Learning_Advanced-customizations-in-deep-learning-and-neural-networks/blob/main/07_keras_custom_loss_metrics_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07: Keras Custom Losses, Functions, and Metrics

**Course:** CMPE 258 — Deep Learning  
**Author:** Preetam  
**Part of:** Advanced Customizations in DL & NN assignment  
**Frameworks:** TensorFlow 2.16 / Keras 3.3  
**Companion video:** TBD

## What this notebook covers
- Custom Huber function and class-based loss
- Custom activation/initializer/regularizer/constraint
- Custom streaming Huber metric

## Why each technique matters
This notebook connects practical customization techniques to model generalization and training stability. Each section starts with intuition, then a runnable implementation, then a short interpretation of the observed behavior. Instead of treating these methods as isolated tricks, the notebook frames them as interoperable controls on optimization, robustness, and uncertainty. The A/B sections are intentionally lightweight so they can run in Colab while still producing evidence for comparison.


In [ ]:
!pip -q install tensorflow==2.16.1 keras==3.3.3 scikit-learn
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Python: 3.11.9
Platform: macOS-26.0.1-arm64-arm-64bit


In [ ]:
# Set deterministic seeds for reproducibility.
import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
import keras

tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)


California Housing is a regression task where Huber-style robustness to outliers is meaningful.


## Custom Huber loss: function closure and serializable class


In [ ]:
import keras
import tensorflow as tf
from keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

X, y = fetch_california_housing(return_X_y=True)
X = X.astype("float32")
y = y.astype("float32")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

def huber_fn(threshold=1.0):
    def loss(y_true, y_pred):
        err = y_true - y_pred
        is_small = tf.abs(err) <= threshold
        small = 0.5 * tf.square(err)
        large = threshold * tf.abs(err) - 0.5 * threshold**2
        return tf.where(is_small, small, large)
    return loss

class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
    def call(self, y_true, y_pred):
        err = y_true - y_pred
        is_small = tf.abs(err) <= self.threshold
        small = 0.5 * tf.square(err)
        large = self.threshold * tf.abs(err) - 0.5 * self.threshold**2
        return tf.where(is_small, small, large)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"threshold": self.threshold})
        return cfg


## Custom activation, initializer, regularizer, and constraint


In [ ]:
def my_leaky_relu(z):
    return tf.maximum(0.2 * z, z)

def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2.0 / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)

def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))

class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self, factor=0.01):
        self.factor = factor
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(self.factor * weights))
    def get_config(self):
        return {"factor": self.factor}

class MyPositiveConstraint(keras.constraints.Constraint):
    def __call__(self, w):
        return tf.where(w < 0.0, tf.zeros_like(w), w)

class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self, factor=0.01):
        self.factor = factor
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(self.factor * weights))
    def get_config(self):
        return {"factor": self.factor}

def my_positive_weights(weights):
    return tf.where(weights < 0.0, tf.zeros_like(weights), weights)


## Custom metric: streaming Huber


In [ ]:
class HuberMetric(keras.metrics.Metric):
    def __init__(self, threshold=1.0, name="huber_metric", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        err = y_true - y_pred
        is_small = tf.abs(err) <= self.threshold
        small = 0.5 * tf.square(err)
        large = self.threshold * tf.abs(err) - 0.5 * self.threshold**2
        values = tf.where(is_small, small, large)
        self.total.assign_add(tf.reduce_sum(values))
        self.count.assign_add(tf.cast(tf.size(values), tf.float32))
    def result(self):
        return self.total / tf.maximum(self.count, 1.0)
    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"threshold": self.threshold})
        return cfg

def make_model():
    return keras.Sequential([
        layers.Input(shape=(Xtr.shape[1],)),
        layers.Dense(64, activation=my_leaky_relu, kernel_initializer=my_glorot_initializer,
                     kernel_regularizer=MyL1Regularizer(0.01), kernel_constraint=MyPositiveConstraint()),
        layers.Dense(1),
    ])

m_mse = make_model()
m_huber = make_model()
m_mse.compile(optimizer="adam", loss="mse", metrics=["mae"])
m_huber.compile(optimizer="adam", loss=HuberLoss(1.0), metrics=["mae", HuberMetric(1.0)])
h_mse = m_mse.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=5, batch_size=64, verbose=0)
h_huber = m_huber.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=5, batch_size=64, verbose=0)

wmin = float(np.min(m_huber.layers[0].get_weights()[0]))
print("Min constrained weight:", wmin)
print("Final MAE MSE:", h_mse.history["val_mae"][-1], "Final MAE Huber:", h_huber.history["val_mae"][-1])
print("Final custom Huber metric:", h_huber.history["val_huber_metric"][-1])


Min constrained weight: 0.0
Final MAE MSE: 0.6694830060005188 Final MAE Huber: 3.048116683959961
Final custom Huber metric: 2.5631465911865234


## Final summary table

| Component | Validation indicator | notes |
|---|---|---|
| Functional Huber | used in compile-ready closure | Easy for quick experimentation |
| Class HuberLoss | serializable via `get_config` | Better for saving/loading |
| Custom activation/initializer/regularizer/constraint | model trains with all four simultaneously | Demonstrates composability |
| HuberMetric | tracked each epoch | Should align with Huber loss trend |

Huber-based training is typically more stable under target outliers than plain MSE. Custom function hooks also make it easier to encode domain assumptions directly in the model.
